# R3 / R6 / R8: Statistical Tests, Parse Failures & Rationale Quality

**Lightweight analysis notebook (CPU-only).** Reads outputs from R1 (3 per-seed runs) and R2 (ablation).

| Task | What | Output |
|------|------|--------|
| R3 | Paired bootstrap significance test (RSQwen vs Vanilla) | p-values for F1-micro & F1-macro |
| R6 | Parse failure analysis: invalid model outputs | Parse failure rate, parseable-only F1 |
| R8 | Rationale quality: sample 50 for human review | `rationale_review_template.csv` |

**Prerequisites — upload all 7 files to `MyDrive/ViTHSD/outputs/` on Drive before running:**
- `predictions_seed_42.json`, `predictions_seed_123.json`, `predictions_seed_456.json`
- `results_seed_42.json`, `results_seed_123.json`, `results_seed_456.json`
- `results_ablations.json`

**Also required in `MyDrive/ViTHSD/src/`:** `config.py`, `data_preparation.py`, `evaluation.py`, `models.py`


## Step 1: Setup

In [1]:
import json, os, sys, random, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

# Change these if not on Colab
USE_COLAB = True

if USE_COLAB:
  from google.colab import drive
  drive.mount('/content/drive')
  DRIVE_BASE = Path('/content/drive/MyDrive/ViTHSD')
else:
  DRIVE_BASE = Path('.') # local

OUTPUT_DIR = DRIVE_BASE / 'outputs'
DATA_DIR = DRIVE_BASE / 'data'

# Copy src files to a local working directory so imports work on Colab
WORK_DIR = Path('/content/vithsd')
WORK_DIR.mkdir(exist_ok=True)
for _f in ['config.py', 'data_preparation.py', 'evaluation.py', 'models.py']:
  _src = DRIVE_BASE / 'src' / _f
  if _src.exists():
    shutil.copy2(_src, WORK_DIR / _f)
    print(f'  Copied {_f}')
  else:
    print(f'  WARNING: {_src} not found')
sys.path.insert(0, str(WORK_DIR))

FINAL_LABELS = [
  'Normal',
  'individuals#offensive', 'individuals#hate',
  'groups#offensive', 'groups#hate',
  'religion#offensive', 'religion#hate',
  'race#offensive', 'race#hate',
  'politics#offensive', 'politics#hate'
]

print(f'\nOutput dir: {OUTPUT_DIR}')
print(f'Files: {[f.name for f in OUTPUT_DIR.iterdir() if f.is_file()]}')


Mounted at /content/drive
  Copied config.py
  Copied data_preparation.py
  Copied evaluation.py
  Copied models.py

Output dir: /content/drive/MyDrive/ViTHSD/outputs
Files: ['results_ablations.json', 'results_datasetA_visobert_rationale_attn.json', 'results_seed_42.json', 'predictions_seed_42.json', 'results_seed_123.json', 'predictions_seed_123.json', 'predictions_seed_456.json', 'results_seed_456.json']


In [2]:
# Load test labels via load_dataset_A (ground truth not saved by per-seed notebooks)
import config as _cfg
_cfg.DATA_DIR = DATA_DIR
from data_preparation import load_dataset_A
_, test_labels = load_dataset_A('test', data_dir=DATA_DIR)
print(f'Test labels loaded: {len(test_labels)} samples')

# Merge per-seed predictions from R1 (6 files -> all_predictions dict)
all_predictions = {}
for _seed in [42, 123, 456]:
  _path = OUTPUT_DIR / f'predictions_seed_{_seed}.json'
  assert _path.exists(), f'Missing: {_path.name} -- upload to Drive/outputs/ first!'
  with open(_path, 'r', encoding='utf-8') as _f:
    _data = json.load(_f)
  all_predictions[f'rsqwen_seed{_seed}'] = _data['rsqwen']
  all_predictions[f'vanilla_seed{_seed}'] = _data['vanilla']
  print(f'  Loaded predictions_seed_{_seed}.json')

# Add ablation predictions from R2 (for R6 parse analysis)
_abl_path = OUTPUT_DIR / 'results_ablations.json'
assert _abl_path.exists(), f'Missing: {_abl_path.name} -- upload to Drive/outputs/ first!'
with open(_abl_path, 'r', encoding='utf-8') as _f:
  ablation_data = json.load(_f)
all_predictions['shuffled_rationale'] = ablation_data['shuffled_predictions']
all_predictions['plain_continuation'] = ablation_data['plain_predictions']
print(f'  Loaded ablation predictions from results_ablations.json')

# Load per-seed metrics for aggregation (Step 2b)
per_seed_results = {}
for _seed in [42, 123, 456]:
  _path = OUTPUT_DIR / f'results_seed_{_seed}.json'
  assert _path.exists(), f'Missing: {_path.name} -- upload to Drive/outputs/ first!'
  with open(_path, 'r', encoding='utf-8') as _f:
    per_seed_results[_seed] = json.load(_f)
  print(f'  Loaded results_seed_{_seed}.json')

multiseed_results = {}  # placeholder (unused by downstream analysis cells)

# Load rationale data
with open(DATA_DIR / 'dataset_rationale.json', 'r', encoding='utf-8') as _f:
  rationale_data = json.load(_f)

print(f'\nPrediction sets ({len(all_predictions)}): {list(all_predictions.keys())}')
print(f'Test samples: {len(test_labels)}')
print(f'Rationale entries: {len(rationale_data)}')


[ViHSDLoader] Loaded 1800 samples from test
Test labels loaded: 1800 samples
  Loaded predictions_seed_42.json
  Loaded predictions_seed_123.json
  Loaded predictions_seed_456.json
  Loaded ablation predictions from results_ablations.json
  Loaded results_seed_42.json
  Loaded results_seed_123.json
  Loaded results_seed_456.json

Prediction sets (8): ['rsqwen_seed42', 'vanilla_seed42', 'rsqwen_seed123', 'vanilla_seed123', 'rsqwen_seed456', 'vanilla_seed456', 'shuffled_rationale', 'plain_continuation']
Test samples: 1800
Rationale entries: 1744


## Step 2b: Aggregate Multi-Seed Results (mean ± std)


In [3]:
METRICS = ['f1_micro', 'f1_macro', 'precision_micro', 'recall_micro', 'subset_accuracy']
SEEDS = [42, 123, 456]

def agg(model_key):
  vals = {m: [] for m in METRICS}
  for s in SEEDS:
    mdata = per_seed_results[s][model_key]
    for m in METRICS:
      vals[m].append(mdata[m])
  return {m: (float(np.mean(vals[m])), float(np.std(vals[m]))) for m in METRICS}

agg_rs  = agg('rsqwen')
agg_van = agg('vanilla')

print('=' * 70)
print('MULTI-SEED RESULTS (mean ± std over seeds 42 / 123 / 456)')
print('=' * 70)
print(f'{"Metric":>22} {"RSQwen":>20} {"Vanilla":>20}')
print('-' * 65)
for m in METRICS:
  rs_m, rs_s = agg_rs[m]
  va_m, va_s = agg_van[m]
  print(f'{m:>22}  {rs_m*100:6.2f} ± {rs_s*100:.2f}%   {va_m*100:6.2f} ± {va_s*100:.2f}%')

print(f'\n{"Per-seed F1-Micro":^60}')
print(f'{"Seed":>8} {"RSQwen":>12} {"Vanilla":>12}')
for s in SEEDS:
  rs_val = per_seed_results[s]['rsqwen']['f1_micro']
  va_val = per_seed_results[s]['vanilla']['f1_micro']
  print(f'{s:>8}  {rs_val*100:9.2f}%  {va_val*100:9.2f}%')

print(f'\n{"Ablation (seed=42)":^60}')
for key, label in [('shuffled_rationale', 'Shuffled-Rationale'),
                   ('plain_continuation',  'Plain-Continuation')]:
  m_val = ablation_data[key]['f1_micro']
  print(f'{label:>22}: F1-Micro = {m_val*100:.2f}%')


MULTI-SEED RESULTS (mean ± std over seeds 42 / 123 / 456)
                Metric               RSQwen              Vanilla
-----------------------------------------------------------------
              f1_micro   55.51 ± 1.27%    59.67 ± 1.08%
              f1_macro   35.70 ± 3.59%    32.95 ± 1.33%
       precision_micro   50.91 ± 2.18%    61.02 ± 1.38%
          recall_micro   61.11 ± 0.91%    58.43 ± 1.79%
       subset_accuracy   48.37 ± 3.46%    56.37 ± 1.37%

                     Per-seed F1-Micro                      
    Seed       RSQwen      Vanilla
      42      53.79%      58.37%
     123      55.94%      59.63%
     456      56.80%      61.02%

                     Ablation (seed=42)                     
    Shuffled-Rationale: F1-Micro = 58.38%
    Plain-Continuation: F1-Micro = 36.35%


---
## R3: Paired Bootstrap Significance Test

Tests whether RSQwen is significantly better than Vanilla Qwen.
Uses paired bootstrap resampling (Koehn 2004) with 10K iterations.

In [4]:
from sklearn.metrics import f1_score

def labels_to_binary(labels_list, all_labels):
  """Convert list of label strings to binary matrix."""
  result = np.zeros((len(labels_list), len(all_labels)), dtype=int)
  for i, labels in enumerate(labels_list):
    for lbl in labels:
      if lbl in all_labels:
        result[i, all_labels.index(lbl)] = 1
  return result


def paired_bootstrap_test(y_true, y_pred_a, y_pred_b, metric_fn,
              n_iterations=10000, seed=42):
  """
  Paired bootstrap significance test.
  H0: system A is no better than system B.
  Returns: p-value (fraction of bootstrap samples where B >= A).
  """
  rng = np.random.RandomState(seed)
  n = len(y_true)
  score_a = metric_fn(y_true, y_pred_a)
  score_b = metric_fn(y_true, y_pred_b)
  observed_diff = score_a - score_b

  count_b_wins = 0
  for _ in range(n_iterations):
    indices = rng.choice(n, size=n, replace=True)
    boot_true = y_true[indices]
    boot_a = y_pred_a[indices]
    boot_b = y_pred_b[indices]
    diff = metric_fn(boot_true, boot_a) - metric_fn(boot_true, boot_b)
    if diff <= 0:
      count_b_wins += 1

  p_value = count_b_wins / n_iterations
  return {
    'score_a': score_a, 'score_b': score_b,
    'observed_diff': observed_diff,
    'p_value': p_value, 'n_iterations': n_iterations,
    'significant_0.05': p_value < 0.05,
    'significant_0.01': p_value < 0.01
  }

print('Bootstrap test function defined')

Bootstrap test function defined


In [5]:
y_true_bin = labels_to_binary(test_labels, FINAL_LABELS)

def f1_micro_fn(y_true, y_pred):
  return f1_score(y_true, y_pred, average='micro', zero_division=0)

def f1_macro_fn(y_true, y_pred):
  return f1_score(y_true, y_pred, average='macro', zero_division=0)

# Run per-seed tests
seeds = [42, 123, 456]
bootstrap_results = {}

for seed in seeds:
  rs_key = f'rsqwen_seed{seed}'
  van_key = f'vanilla_seed{seed}'

  if rs_key not in all_predictions or van_key not in all_predictions:
    print(f'Skipping seed {seed}: predictions not found')
    continue

  rs_bin = labels_to_binary(all_predictions[rs_key], FINAL_LABELS)
  van_bin = labels_to_binary(all_predictions[van_key], FINAL_LABELS)

  print(f'\n--- Seed {seed} ---')
  res_micro = paired_bootstrap_test(y_true_bin, rs_bin, van_bin, f1_micro_fn)
  res_macro = paired_bootstrap_test(y_true_bin, rs_bin, van_bin, f1_macro_fn)

  print(f' F1-Micro: RSQwen={res_micro["score_a"]:.4f} vs Vanilla={res_micro["score_b"]:.4f}')
  print(f'  diff={res_micro["observed_diff"]:.4f}, p={res_micro["p_value"]:.4f} {"*" if res_micro["significant_0.05"] else "n.s."}')
  print(f' F1-Macro: RSQwen={res_macro["score_a"]:.4f} vs Vanilla={res_macro["score_b"]:.4f}')
  print(f'  diff={res_macro["observed_diff"]:.4f}, p={res_macro["p_value"]:.4f} {"*" if res_macro["significant_0.05"] else "n.s."}')

  bootstrap_results[f'seed_{seed}'] = {
    'f1_micro': res_micro, 'f1_macro': res_macro
  }


--- Seed 42 ---
 F1-Micro: RSQwen=0.4577 vs Vanilla=0.4214
  diff=0.0362, p=0.0037 *
 F1-Macro: RSQwen=0.2421 vs Vanilla=0.2450
  diff=-0.0030, p=0.5785 n.s.

--- Seed 123 ---
 F1-Micro: RSQwen=0.4576 vs Vanilla=0.4534
  diff=0.0042, p=0.3769 n.s.
 F1-Macro: RSQwen=0.3241 vs Vanilla=0.2532
  diff=0.0709, p=0.0004 *

--- Seed 456 ---
 F1-Micro: RSQwen=0.4875 vs Vanilla=0.4613
  diff=0.0262, p=0.0186 *
 F1-Macro: RSQwen=0.3002 vs Vanilla=0.2754
  diff=0.0247, p=0.0307 *


In [6]:
# Pooled test: concatenate predictions across all seeds
rs_pooled, van_pooled, true_pooled = [], [], []

for seed in seeds:
  rs_key = f'rsqwen_seed{seed}'
  van_key = f'vanilla_seed{seed}'
  if rs_key in all_predictions and van_key in all_predictions:
    rs_pooled.append(labels_to_binary(all_predictions[rs_key], FINAL_LABELS))
    van_pooled.append(labels_to_binary(all_predictions[van_key], FINAL_LABELS))
    true_pooled.append(y_true_bin)

if rs_pooled:
  rs_cat = np.concatenate(rs_pooled)
  van_cat = np.concatenate(van_pooled)
  true_cat = np.concatenate(true_pooled)

  print('--- Pooled (all seeds concatenated) ---')
  pooled_micro = paired_bootstrap_test(true_cat, rs_cat, van_cat, f1_micro_fn)
  pooled_macro = paired_bootstrap_test(true_cat, rs_cat, van_cat, f1_macro_fn)

  print(f' F1-Micro: diff={pooled_micro["observed_diff"]:.4f}, p={pooled_micro["p_value"]:.4f}')
  print(f' F1-Macro: diff={pooled_macro["observed_diff"]:.4f}, p={pooled_macro["p_value"]:.4f}')

  bootstrap_results['pooled'] = {
    'f1_micro': pooled_micro, 'f1_macro': pooled_macro
  }

print('\nR3 complete!')

--- Pooled (all seeds concatenated) ---
 F1-Micro: diff=0.0214, p=0.0038
 F1-Macro: diff=0.0369, p=0.0008

R3 complete!


---
## R6: Parse Failure Analysis

Check how many model outputs fail to parse into valid labels.
Compute: failure rate, parseable-only F1, failure examples.

In [7]:
def check_parse_validity(predictions, valid_labels):
  """Check each prediction for invalid labels."""
  valid_set = set(valid_labels)
  results = []
  for i, pred in enumerate(predictions):
    invalid = [l for l in pred if l not in valid_set]
    empty = len(pred) == 0
    results.append({
      'index': i,
      'predicted': pred,
      'invalid_labels': invalid,
      'is_empty': empty,
      'is_valid': len(invalid) == 0 and not empty
    })
  return results


def analyze_parse_failures(all_predictions, test_labels, valid_labels):
  """Comprehensive parse failure analysis across all prediction sets."""
  analysis = {}
  y_true_bin = labels_to_binary(test_labels, valid_labels)

  for model_key, preds in all_predictions.items():
    validity = check_parse_validity(preds, valid_labels)
    n_total = len(preds)
    n_valid = sum(1 for v in validity if v['is_valid'])
    n_empty = sum(1 for v in validity if v['is_empty'])
    n_invalid = n_total - n_valid

    # Collect invalid label types
    all_invalid = []
    for v in validity:
      all_invalid.extend(v['invalid_labels'])
    invalid_counter = Counter(all_invalid)

    # Parseable-only F1
    valid_indices = [v['index'] for v in validity if v['is_valid']]
    if valid_indices:
      valid_true = y_true_bin[valid_indices]
      valid_pred = labels_to_binary([preds[i] for i in valid_indices], valid_labels)
      f1_mi = f1_score(valid_true, valid_pred, average='micro', zero_division=0)
      f1_ma = f1_score(valid_true, valid_pred, average='macro', zero_division=0)
    else:
      f1_mi = f1_ma = 0.0

    # Full F1 (treating invalid as empty = all-zero)
    full_pred = labels_to_binary(preds, valid_labels)
    f1_mi_full = f1_score(y_true_bin, full_pred, average='micro', zero_division=0)
    f1_ma_full = f1_score(y_true_bin, full_pred, average='macro', zero_division=0)

    # Failure examples (up to 5)
    failures = [v for v in validity if not v['is_valid']][:5]

    analysis[model_key] = {
      'total': n_total,
      'valid': n_valid,
      'invalid': n_invalid,
      'empty': n_empty,
      'failure_rate': n_invalid / n_total if n_total > 0 else 0,
      'invalid_label_counts': dict(invalid_counter.most_common(10)),
      'f1_micro_parseable_only': f1_mi,
      'f1_macro_parseable_only': f1_ma,
      'f1_micro_full': f1_mi_full,
      'f1_macro_full': f1_ma_full,
      'f1_micro_gap': f1_mi - f1_mi_full,
      'failure_examples': failures
    }

  return analysis

print('Parse analysis functions defined')

Parse analysis functions defined


In [8]:
from sklearn.metrics import f1_score

parse_analysis = analyze_parse_failures(all_predictions, test_labels, FINAL_LABELS)

print(f'{"="*75}')
print(f'{"PARSE FAILURE ANALYSIS":^75}')
print(f'{"="*75}')
print(f'{"Model":>25} {"Total":>6} {"Valid":>6} {"Fail":>6} {"Rate":>8} {"F1-μ(full)":>10} {"F1-μ(parse)":>11}')
print(f'{"-"*75}')

for model_key, info in parse_analysis.items():
  print(f'{model_key:>25} {info["total"]:>6} {info["valid"]:>6} {info["invalid"]:>6} '
     f'{info["failure_rate"]*100:>7.2f}% {info["f1_micro_full"]*100:>9.2f}% {info["f1_micro_parseable_only"]*100:>10.2f}%')

# Show common invalid labels
print(f'\n--- Most Common Invalid Labels ---')
for model_key, info in parse_analysis.items():
  if info['invalid_label_counts']:
    print(f'{model_key}: {info["invalid_label_counts"]}')

                          PARSE FAILURE ANALYSIS                           
                    Model  Total  Valid   Fail     Rate F1-μ(full) F1-μ(parse)
---------------------------------------------------------------------------
            rsqwen_seed42   1800   1205    595   33.06%     45.77%      46.86%
           vanilla_seed42   1800    728   1072   59.56%     42.14%      52.12%
           rsqwen_seed123   1800   1076    724   40.22%     45.76%      47.94%
          vanilla_seed123   1800    767   1033   57.39%     45.34%      54.13%
           rsqwen_seed456   1800   1167    633   35.17%     48.75%      50.50%
          vanilla_seed456   1800    712   1088   60.44%     46.13%      57.25%
       shuffled_rationale   1800   1115    685   38.06%     48.67%      50.66%
       plain_continuation   1800   1732     68    3.78%     41.66%      41.70%

--- Most Common Invalid Labels ---
rsqwen_seed42: {'normal': 595}
vanilla_seed42: {'normal': 1072}
rsqwen_seed123: {'normal': 724}
vanil

---
## R8: Rationale Quality — Sample & Human Review Template

Stratified sample of 50 rationale entries for human annotation.
Two reviewers score each on 3 criteria (1-5 Likert):
1. **Relevance** — Does rationale address the hate speech indicators?
2. **Faithfulness** — Is implied statement accurate given the text?
3. **Completeness** — Are all relevant aspects covered?

Inter-annotator agreement via Cohen's kappa.

In [9]:
random.seed(42)

# Filter to train samples with valid rationale
valid_rationale = []
for r in rationale_data:
  if str(r.get('dataset', '')).lower().strip() != 'train':
    continue
  content = (r.get('content') or '').strip()
  implied = (r.get('implied_statement') or '').strip()
  rats = r.get('rationale', [])
  if content and implied and rats:
    labels = r.get('labels', [])
    is_hateful = any('hate' in l for l in labels)
    valid_rationale.append({
      'id': r.get('id', ''),
      'content': content,
      'implied_statement': implied,
      'rationale': ' | '.join(rats) if isinstance(rats, list) else str(rats),
      'labels': ', '.join(labels),
      'is_hateful': is_hateful
    })

# Stratified sample: 50% hateful, 50% offensive-only
hateful = [r for r in valid_rationale if r['is_hateful']]
non_hateful = [r for r in valid_rationale if not r['is_hateful']]

n_sample = 50
n_hate = min(n_sample // 2, len(hateful))
n_non = min(n_sample - n_hate, len(non_hateful))
n_hate = n_sample - n_non # adjust if needed

sample = random.sample(hateful, n_hate) + random.sample(non_hateful, n_non)
random.shuffle(sample)

print(f'Total valid: {len(valid_rationale)} (hateful: {len(hateful)}, non-hateful: {len(non_hateful)})')
print(f'Sampled: {len(sample)} ({n_hate} hateful, {n_non} non-hateful)')

Total valid: 1221 (hateful: 784, non-hateful: 437)
Sampled: 50 (25 hateful, 25 non-hateful)


In [10]:
# Create review CSV template
rows = []
for i, s in enumerate(sample, 1):
  rows.append({
    'sample_id': i,
    'original_id': s['id'],
    'content': s['content'],
    'labels': s['labels'],
    'implied_statement': s['implied_statement'],
    'rationale': s['rationale'],
    'reviewer1_relevance': '',
    'reviewer1_faithfulness': '',
    'reviewer1_completeness': '',
    'reviewer2_relevance': '',
    'reviewer2_faithfulness': '',
    'reviewer2_completeness': '',
    'notes': ''
  })

df_review = pd.DataFrame(rows)

csv_path = OUTPUT_DIR / 'rationale_review_template.csv'
df_review.to_csv(csv_path, index=False, encoding='utf-8-sig')

print(f'Review template: {csv_path}')
print(f'Columns: {list(df_review.columns)}')
print(f'\nScoring guide: 1=Poor, 2=Below avg, 3=Adequate, 4=Good, 5=Excellent')
print('Each reviewer fills their 3 columns for all 50 samples.')
df_review.head(2)

Review template: /content/drive/MyDrive/ViTHSD/outputs/rationale_review_template.csv
Columns: ['sample_id', 'original_id', 'content', 'labels', 'implied_statement', 'rationale', 'reviewer1_relevance', 'reviewer1_faithfulness', 'reviewer1_completeness', 'reviewer2_relevance', 'reviewer2_faithfulness', 'reviewer2_completeness', 'notes']

Scoring guide: 1=Poor, 2=Below avg, 3=Adequate, 4=Good, 5=Excellent
Each reviewer fills their 3 columns for all 50 samples.


,sample_id,original_id,content,labels,implied_statement,rationale,reviewer1_relevance,reviewer1_faithfulness,reviewer1_completeness,reviewer2_relevance,reviewer2_faithfulness,reviewer2_completeness,notes
0,1,168,Đmcs,"race#hate, politics#hate",cộng sản đáng ghét,"Cụm từ ""Đmcs"" được sử dụng với mục đích xúc ph...",,,,,,,
1,2,2808,Iu cô gái khơ me hàm răng trắng tác cái l** đe...,individuals#hate,phụ nữ bị tình dục hóa,Việc sử dụng từ ngữ miệt thị như 'l** đen' nhằ...,,,,,,,


In [11]:
def compute_cohens_kappa(ratings1, ratings2):
  """Cohen's kappa for ordinal ratings."""
  assert len(ratings1) == len(ratings2)
  n = len(ratings1)
  categories = sorted(set(ratings1) | set(ratings2))
  k = len(categories)
  cat_idx = {c: i for i, c in enumerate(categories)}

  conf = np.zeros((k, k), dtype=float)
  for r1, r2 in zip(ratings1, ratings2):
    conf[cat_idx[r1], cat_idx[r2]] += 1

  po = np.trace(conf) / n # observed agreement
  row_sums = conf.sum(axis=1)
  col_sums = conf.sum(axis=0)
  pe = np.sum(row_sums * col_sums) / (n * n) # expected agreement

  if pe == 1.0:
    return 1.0
  kappa = (po - pe) / (1 - pe)
  return kappa


def analyze_completed_review(csv_path):
  """Run after reviewers fill the template."""
  df = pd.read_csv(csv_path)
  criteria = ['relevance', 'faithfulness', 'completeness']

  print(f'{"="*50}')
  print(f'{"INTER-ANNOTATOR AGREEMENT":^50}')
  print(f'{"="*50}')

  kappas = {}
  for c in criteria:
    r1 = df[f'reviewer1_{c}'].dropna().astype(int).tolist()
    r2 = df[f'reviewer2_{c}'].dropna().astype(int).tolist()
    if len(r1) == len(r2) and len(r1) > 0:
      k = compute_cohens_kappa(r1, r2)
      kappas[c] = k
      print(f' {c:>20}: kappa = {k:.3f}')
    else:
      print(f' {c:>20}: INCOMPLETE ({len(r1)} vs {len(r2)} ratings)')

  # Average scores per reviewer
  print(f'\n{"MEAN SCORES":^50}')
  for c in criteria:
    for rev in [1, 2]:
      col = f'reviewer{rev}_{c}'
      vals = df[col].dropna().astype(float)
      if len(vals) > 0:
        print(f' R{rev} {c:>15}: {vals.mean():.2f} +/- {vals.std():.2f}')

  return kappas

print('Cohen kappa and review analysis functions defined.')
print('\nAfter reviewers complete the CSV, run:')
print(' kappas = analyze_completed_review(OUTPUT_DIR / "rationale_review_template.csv")')

Cohen kappa and review analysis functions defined.

After reviewers complete the CSV, run:
 kappas = analyze_completed_review(OUTPUT_DIR / "rationale_review_template.csv")


---
## Save All Analysis Results

In [12]:
def convert_to_serializable(obj):
  if isinstance(obj, dict):
    return {k: convert_to_serializable(v) for k, v in obj.items()}
  elif isinstance(obj, (list, tuple)):
    return [convert_to_serializable(item) for item in obj]
  elif isinstance(obj, (np.integer, np.floating)):
    return float(obj)
  elif isinstance(obj, np.ndarray):
    return obj.tolist()
  return obj

analysis_results = {
  'R3_bootstrap': convert_to_serializable(bootstrap_results),
  'R6_parse_failures': convert_to_serializable(parse_analysis),
  'R8_sample_info': {
    'total_valid': len(valid_rationale),
    'sampled': len(sample),
    'n_hateful': n_hate,
    'n_non_hateful': n_non,
    'review_csv': str(csv_path)
  }
}

with open(OUTPUT_DIR / 'analysis_results.json', 'w') as f:
  json.dump(analysis_results, f, indent=2, ensure_ascii=False)

print(f'Saved: {OUTPUT_DIR / "analysis_results.json"}')
print(f'Saved: {OUTPUT_DIR / "rationale_review_template.csv"}')
print('\nAll analysis complete!')

Saved: /content/drive/MyDrive/ViTHSD/outputs/analysis_results.json
Saved: /content/drive/MyDrive/ViTHSD/outputs/rationale_review_template.csv

All analysis complete!
